# WTI Hourly Price Data — Yahoo Finance

This notebook downloads two years of hourly OHLCV (Open, High, Low, Close, Volume) 
data for WTI Crude Oil futures (ticker: CL=F) from Yahoo Finance using the yfinance 
library. It then computes four derived liquidity metrics — log volume, price range, 
log return, and the Amihud illiquidity ratio — which serve as the primary dependent 
variables in the thesis analysis. The resulting dataset is saved to 
`01_data/raw/price/wti_hourly_raw.csv`.

### Necessary imports

In [8]:
import yfinance as yf
import pandas as pd
import numpy as np
import sqlite3

### Connect to the db

In [9]:
conn = sqlite3.connect("../01_data/wti_thesis.db")
print("Connected to wti_thesis.db")

Connected to wti_thesis.db


### Obtain DXY, VIX information

In [10]:
tickers = {
    "CL=F": "wti",
    "DX-Y.NYB": "dxy",
    "^VIX": "vix"
}

data = {}
for ticker, name in tickers.items():
    df = yf.download(ticker, period="2y", interval="1h", progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index = pd.to_datetime(df.index).tz_convert('UTC')
    df.index.name = 'datetime_hour'
    data[name] = df
    print(f"{name.upper()}: {len(df)} records | {df.index.min().date()} to {df.index.max().date()}")

WTI: 11224 records | 2024-05-12 to 2026-05-12
DXY: 11939 records | 2024-05-12 to 2026-05-12
VIX: 7114 records | 2024-05-13 to 2026-05-12


### Obtain liquidity information

- **Open:** Price at th emoment of opening the time window.

- **High:** Highest price in the time window.

- **Low:** Lowest price in the time window.

- **Volume:** Amount of contracts on the time window.

- **Close:** Price at the moment of closing the time window.

In [11]:
df_price = yf.download("CL=F", period="2y", interval="1h", progress=False)

# If Yahoo Finance changes the order of the columns in the future, this avoid silent erros of calculating values with different columns
if isinstance(df_price.columns, pd.MultiIndex):
    df_price.columns = df_price.columns.get_level_values(0)
df_price.index = pd.to_datetime(df_price.index).tz_convert('UTC')
df_price.index.name = 'Datetime'

print(f"Registros: {len(df_price)}")
print(f"Rango: {df_price.index.min()} a {df_price.index.max()}")
print(df_price.tail(5))

Registros: 11224
Rango: 2024-05-12 22:00:00+00:00 a 2026-05-12 13:00:00+00:00
Price                           Close        High         Low        Open  \
Datetime                                                                    
2026-05-12 09:00:00+00:00  101.089996  101.760002  100.860001  100.919998   
2026-05-12 10:00:00+00:00  101.669998  101.970001  100.820000  101.080002   
2026-05-12 11:00:00+00:00  101.269997  102.050003  101.150002  101.660004   
2026-05-12 12:00:00+00:00  100.769997  101.760002  100.599998  101.279999   
2026-05-12 13:00:00+00:00  101.260002  101.370003  100.669998  100.760002   

Price                      Volume  
Datetime                           
2026-05-12 09:00:00+00:00    6451  
2026-05-12 10:00:00+00:00    5754  
2026-05-12 11:00:00+00:00    6163  
2026-05-12 12:00:00+00:00   12298  
2026-05-12 13:00:00+00:00    2901  


### Calculate liquidity variables derived from the OHLCV entries

Four liquidity metrics are calculated from the raw OHLCV data:

- **log_volume:** 
  - Directly measures trading activity — how many contracts changed hands
  - Log transformation handles the right-skewed distribution correctly
  - This IS liquidity in the most direct sense for futures markets
  - Standard in market microstructure literature

- **price_range:**
  - High − Low within the hour captures intraday volatility
  - Wider range = more price uncertainty = less liquidity (wider bid-ask spreads)
  - Parkinson (1980) is a well-cited estimator, defensible academically
  - Correlated with bid-ask spread which we can't get from yfinance

- **log_return:** log(Close_t / Close_{t-1}) — the hourly price return.
  - No liquidity variable, but price variable, measures how much the priced moved, not how easy it was to trade. This is not a dependent variable

- **amihud:** 
  - |log_return| / Volume = price impact per unit of volume
  - High Amihud = prices move a lot per contract traded = illiquid market
  - Amihud (2002) is arguably the most cited liquidity measure in finance literature
  - Captures the market impact dimension of liquidity


In [12]:
# Cell 4 — Compute WTI liquidity features
df_wti = data['wti'].copy()
df_wti['log_volume'] = np.log(df_wti['Volume'] + 1)
df_wti['price_range'] = df_wti['High'] - df_wti['Low']
df_wti['log_return'] = np.log(df_wti['Close'] / df_wti['Close'].shift(1))
df_wti['amihud'] = df_wti['log_return'].abs() / (df_wti['Volume'] + 1)

print("Liquidity features computed:")
print(df_wti[['log_volume', 'price_range', 'amihud']].describe().round(4))

Liquidity features computed:
Price  log_volume  price_range      amihud
count  11224.0000   11224.0000  11223.0000
mean       8.3222       0.4890      0.0001
std        2.1840       0.6176      0.0013
min        0.0000       0.0000      0.0000
25%        7.5147       0.1900      0.0000
50%        8.7252       0.3300      0.0000
75%        9.7692       0.5400      0.0000
max       13.3046      14.4000      0.1107


In [13]:
# Cell 5 — Build market_context and save to DB
df_dxy = data['dxy'][['Close']].rename(columns={'Close': 'dxy'})
df_vix = data['vix'][['Close']].rename(columns={'Close': 'vix'})

df_market = df_wti[['log_volume', 'price_range', 'log_return', 'amihud']].copy()
df_market = df_market.join(df_dxy, how='left')
df_market = df_market.join(df_vix, how='left')
df_market['vix'] = df_market['vix'].ffill(limit=16) # Forward fill so when VIX is null, it carries forward the last known VIX value for up to 16 hours. After 16 hours without a new VIX reading, it leaves null.

# Add empty Cushing column — will be filled in notebook 09
df_market['cushing_inventory'] = None
df_market = df_market.reset_index()
df_market['datetime_hour'] = df_market['datetime_hour'].astype(str)

df_market.to_sql('market_context', conn, if_exists='replace', index=False)

print(f"market_context saved: {len(df_market)} records")
print(f"DXY coverage: {df_market['dxy'].notna().sum()} / {len(df_market)}")
print(f"VIX coverage: {df_market['vix'].notna().sum()} / {len(df_market)}")

market_context saved: 11224 records
DXY coverage: 11217 / 11224
VIX coverage: 11209 / 11224


### Save data

In [14]:
# Cell 6 — Verify
sample = pd.read_sql("""
    SELECT datetime_hour, log_volume, price_range, dxy, vix
    FROM market_context
    WHERE dxy IS NOT NULL
    LIMIT 5
""", conn)
print(sample)

counts = pd.read_sql("""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN dxy IS NOT NULL THEN 1 ELSE 0 END) as dxy_count,
        SUM(CASE WHEN vix IS NOT NULL THEN 1 ELSE 0 END) as vix_count
    FROM market_context
""", conn)
print(counts)

conn.close()
print("Done.")

               datetime_hour  log_volume  price_range         dxy   vix
0  2024-05-12 22:00:00+00:00    7.424762     0.209999  105.306000  None
1  2024-05-12 23:00:00+00:00    6.236370     0.070000  105.330002  None
2  2024-05-13 00:00:00+00:00    7.054450     0.150002  105.347000  None
3  2024-05-13 01:00:00+00:00    8.145260     0.250000  105.349998  None
4  2024-05-13 02:00:00+00:00    7.719130     0.320000  105.315002  None
   total  dxy_count  vix_count
0  11224      11217      11209
Done.
